In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('./input_data/inaugural_speeches.csv')

In [3]:
df.head(2)

,Name,Inaugural Address,Date,text
0,George Washington,First Inaugural Address,"Thursday, April 30, 1789",Fellow-Citizens of the Senate and of the House...
1,George Washington,Second Inaugural Address,"Monday, March 4, 1793",Fellow Citizens: I AM again called upon by th...


In [4]:
df.text[0][:100]

'Fellow-Citizens of the Senate and of the House of Representatives:  AMONG the vicissitudes incident '

### Pre-processing text

In [5]:
df['clean_text'] = df.text.str.replace('[^a-zA-Z]', ' ', regex=True)

In [6]:
df.clean_text = df.clean_text.str.lower()

In [7]:
df.clean_text = df.clean_text.str.replace('\s+', ' ', regex=True)

In [8]:
df.clean_text[0][:100]

'fellow citizens of the senate and of the house of representatives among the vicissitudes incident to'

### Extract info from text

In [9]:
df['char_count'] = df.clean_text.str.len()

In [10]:
df['word_count']  = df.clean_text.str.split().str.len()

In [11]:
df['avg_word_length'] = df.char_count/df.word_count

In [12]:
df['year'] = df.Date.str.extract('(\d{4})')

In [13]:
df_final = df[['Name', 'Date', 'clean_text', 'char_count', 'word_count', 'avg_word_length', 'year']]
df_final.head()

,Name,Date,clean_text,char_count,word_count,avg_word_length,year
0,George Washington,"Thursday, April 30, 1789",fellow citizens of the senate and of the house...,8503,1432,5.937849,1789
1,George Washington,"Monday, March 4, 1793",fellow citizens i am again called upon by the ...,775,135,5.740741,1793
2,John Adams,"Saturday, March 4, 1797",when it was first perceived in early times tha...,13605,2323,5.856651,1797
3,Thomas Jefferson,"Wednesday, March 4, 1801",friends and fellow citizens called upon to und...,9931,1736,5.720622,1801
4,Thomas Jefferson,"Monday, March 4, 1805",proceeding fellow citizens to that qualificati...,12680,2169,5.846012,1805


### CountVectorizer

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

In [15]:
vectorizer = CountVectorizer(min_df=0.2, max_df=0.8)

In [16]:
vectorizer.fit(df_final['clean_text'])

CountVectorizer(max_df=0.8, min_df=0.2)

In [17]:
vectorizer.get_feature_names_out()

array(['abiding', 'ability', 'able', 'about', 'above', 'abroad', 'accept',
       'accomplished', 'achieve', 'across', 'act', 'action', 'acts',
       'add', 'adequate', 'administration', 'adopted', 'advance',
       'advantage', 'affairs', 'afford', 'after', 'again', 'against',
       'age', 'ago', 'agriculture', 'aid', 'alike', 'almighty', 'almost',
       'alone', 'along', 'already', 'also', 'always', 'am', 'america',
       'american', 'americans', 'among', 'ancient', 'another', 'appear',
       'armed', 'arms', 'around', 'ask', 'assume', 'attempt', 'attention',
       'authority', 'avoid', 'away', 'back', 'balance', 'basis', 'bear',
       'because', 'become', 'beginning', 'being', 'belief', 'believe',
       'believed', 'belongs', 'benefit', 'benefits', 'best', 'better',
       'between', 'beyond', 'birth', 'bless', 'blessings', 'blood',
       'body', 'bonds', 'born', 'both', 'bound', 'branches', 'bring',
       'brought', 'build', 'burden', 'burdens', 'business', 'call',
      

In [18]:
vectorized_data = vectorizer.transform(df_final['clean_text'])
vectorized_data

<58x818 sparse matrix of type '<class 'numpy.int64'>'
	with 17406 stored elements in Compressed Sparse Row format>

In [19]:
pd.DataFrame(vectorized_data.toarray(), columns=vectorizer.get_feature_names_out()).add_prefix('c_')

,c_abiding,c_ability,c_able,c_about,c_above,c_abroad,c_accept,c_accomplished,c_achieve,c_across,...,c_women,c_words,c_work,c_wrong,c_year,c_years,c_yet,c_you,c_young,c_your
0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,5,0,9
1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,2,3,0,0,0,1
3,0,0,0,1,1,1,0,0,0,0,...,0,0,1,2,0,0,2,7,0,7
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,2,2,2,4,0,4
5,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
7,0,0,2,1,0,1,0,0,0,0,...,0,0,3,0,0,2,0,0,0,0
8,0,1,1,1,0,0,0,0,0,0,...,0,0,0,0,4,6,1,2,0,0
9,0,1,0,1,0,1,0,1,0,0,...,0,1,1,0,2,6,0,4,0,3


### IF-IDF vectorizer

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [21]:
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')

In [22]:
vectorized_data_w = vectorizer.fit_transform(df.clean_text)

In [23]:
pd.DataFrame(vectorized_data_w.toarray(), columns=vectorizer.get_feature_names_out()).head()

,action,administration,america,american,americans,believe,best,better,change,citizens,...,things,time,today,union,united,war,way,work,world,years
0,0.000000,0.133415,0.000000,0.105388,0.0,0.000000,0.000000,0.000000,0.000000,0.229644,...,0.000000,0.045929,0.0,0.136012,0.203593,0.000000,0.060755,0.000000,0.045929,0.052694
1,0.000000,0.261016,0.266097,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.179712,...,0.000000,0.000000,0.0,0.000000,0.199157,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.092436,0.157058,0.073018,0.0,0.000000,0.026112,0.060460,0.000000,0.106072,...,0.032030,0.021214,0.0,0.062823,0.070529,0.024339,0.000000,0.000000,0.063643,0.073018
3,0.000000,0.092693,0.000000,0.000000,0.0,0.090942,0.117831,0.045471,0.053335,0.223369,...,0.048179,0.000000,0.0,0.094497,0.000000,0.036610,0.000000,0.039277,0.095729,0.000000
4,0.041334,0.039761,0.000000,0.031408,0.0,0.000000,0.067393,0.039011,0.091514,0.273760,...,0.082667,0.164256,0.0,0.121605,0.030338,0.094225,0.000000,0.000000,0.054752,0.062817


In [24]:
from sklearn.feature_extraction.text import TfidfTransformer

In [25]:
vectorizer = CountVectorizer(max_features=100, stop_words='english')

In [26]:
vectorized_data_c = vectorizer.fit_transform(df.clean_text)

In [27]:
transformer = TfidfTransformer()

In [28]:
vectorized_data_ct = transformer.fit_transform(vectorized_data_c)

In [29]:
pd.DataFrame(vectorized_data_ct.toarray(), columns=vectorizer.get_feature_names_out()).head()

,action,administration,america,american,americans,believe,best,better,change,citizens,...,things,time,today,union,united,war,way,work,world,years
0,0.000000,0.133415,0.000000,0.105388,0.0,0.000000,0.000000,0.000000,0.000000,0.229644,...,0.000000,0.045929,0.0,0.136012,0.203593,0.000000,0.060755,0.000000,0.045929,0.052694
1,0.000000,0.261016,0.266097,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.179712,...,0.000000,0.000000,0.0,0.000000,0.199157,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.092436,0.157058,0.073018,0.0,0.000000,0.026112,0.060460,0.000000,0.106072,...,0.032030,0.021214,0.0,0.062823,0.070529,0.024339,0.000000,0.000000,0.063643,0.073018
3,0.000000,0.092693,0.000000,0.000000,0.0,0.090942,0.117831,0.045471,0.053335,0.223369,...,0.048179,0.000000,0.0,0.094497,0.000000,0.036610,0.000000,0.039277,0.095729,0.000000
4,0.041334,0.039761,0.000000,0.031408,0.0,0.000000,0.067393,0.039011,0.091514,0.273760,...,0.082667,0.164256,0.0,0.121605,0.030338,0.094225,0.000000,0.000000,0.054752,0.062817


### Ngram-range

In [30]:
vectorizer = CountVectorizer(max_features=100, stop_words='english', ngram_range=(3,3))

In [31]:
vectorized_data_ngram_3 = vectorizer.fit_transform(df.clean_text)

In [33]:
vectorizer.get_feature_names_out()

array(['ability preserve protect', 'agriculture commerce manufactures',
       'america ideal freedom', 'amity mutual concession',
       'anchor peace home', 'ask bow heads', 'best ability preserve',
       'best interests country', 'bless god bless', 'bless united states',
       'chief justice mr', 'children children children',
       'citizens united states', 'civil religious liberty',
       'civil service reform', 'commerce united states',
       'confidence fellow citizens', 'congress extraordinary session',
       'constitution does expressly', 'constitution united states',
       'coordinate branches government', 'day task people',
       'defend constitution united', 'distinction powers granted',
       'distinguished guests fellow', 'does expressly say',
       'equal exact justice', 'era good feeling',
       'executive branch government', 'faithfully execute office',
       'fellow citizens assembled', 'fellow citizens called',
       'fellow citizens large', 'fellow citiz

In [35]:
df_ngram_3 = pd.DataFrame(vectorized_data_ngram_3.toarray(), columns=vectorizer.get_feature_names_out())
df_ngram_3.head(2)

,ability preserve protect,agriculture commerce manufactures,america ideal freedom,amity mutual concession,anchor peace home,ask bow heads,best ability preserve,best interests country,bless god bless,bless united states,...,thank god bless,turning away old,united states america,united states best,united states government,united states great,united states maintain,united states territory,vice president mr,welfare secure blessings
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [38]:
df_ngram_3.sum().sort_values(ascending=False)

constitution united states    20
people united states          13
mr chief justice              10
preserve protect defend       10
president united states        8
                              ..
distinction powers granted     2
physical moral political       2
mr president vice              2
government peace war           2
mr majority leader             2
Length: 100, dtype: int64